# Jun-2026 LAPD Langmuir IV — one-position inspection

Walks the full IV workflow for **one probe position** so traces and intermediate
results can be checked and knobs tuned before any batch run / saving.

Reader + analysis helpers live in `Jun2026_IV.py`; this notebook just drives them.

**This stage saves nothing** — no `.npz`, no plot files, no batch loop.

Workflow: configure → discover channels → positions → read raw → detect sweeps
→ reshape + smooth → analyze one trace → repeat per probe.


## 1. Configure — set the run file and knobs

In [2]:
import importlib
get_ipython().run_line_magic("matplotlib", "qt")
# get_ipython().run_line_magic("matplotlib", "inline")

import matplotlib.pyplot as plt

import Jun2026_IV as jiv
importlib.reload(jiv)   # re-run this cell after editing Jun2026_IV.py

from data_analysis.io import open_lapd
from data_analysis.plasma.langmuir import find_sweep_indices, reshape_IV, analyze_IV
from scipy.ndimage import gaussian_filter1d

# >>> SET THIS to the run you want to inspect (leave knobs in Jun2026_IV.py) <<<
ifn = r"D:\data\LAPD\jun2026-jia\00-He-800G-bias40V-LP-p29-line_2026-06-10.hdf5"

print(f"RESISTOR = {jiv.RESISTOR}, Aprobe = {jiv.Aprobe}, I_SIGN = {jiv.I_SIGN}")

assert ifn, "Set `ifn` to a run file first."
run = open_lapd(ifn)
print("backend:", run.backend)

RESISTOR = 25.0, Aprobe = 0.002, I_SIGN = -1
backend: pydaq


## 2. Discover I/V channels (all complete probe pairs)

Auto-detects the LP scope group and, from each channel's `description`, groups
channels by tip into complete I+V pairs.  **Every** complete pair is collected
(both probes analyzed); tips missing a channel are flagged and left blank, and
fixed-bias ion-saturation (I-only) tips are not paired.

In [3]:
scope_name = jiv.SCOPE_NAME if jiv.SCOPE_NAME is not None else jiv.find_lp_scope(ifn)
pairs, incomplete = jiv.discover_lp_channels(ifn, scope_name)

# tips to analyze (override forces a single tip; else all complete pairs)
if jiv.I_CHAN is not None and jiv.V_CHAN is not None:
    tips = {"override": {"I": jiv.I_CHAN, "V": jiv.V_CHAN}}
else:
    tips = dict(pairs)

print("\nTips that will be analyzed:")
for tip, ch in tips.items():
    print(f"  tip {tip}: I={ch['I']}, V={ch['V']}")
if incomplete:
    print("Tips left BLANK (incomplete):", list(incomplete))


LP scope 'lpscope' channel descriptions:
  C2: 'V, LP@P29-L'  -> quantity=V, tip=L
  C3: 'I, LP@P29-R'  -> quantity=I, tip=R
  C4: 'V, LP@P29-R'  -> quantity=V, tip=R

Complete I+V tips (will be analyzed):
  tip R: I=C3, V=C4

*** FLAG: tips missing a channel (NOT analyzed) ***
  tip L: have [V=C2], MISSING I (e.g. LAPD_DAQ dropped-C1 bug)

Tips that will be analyzed:
  tip R: I=C3, V=C4
Tips left BLANK (incomplete): ['L']


## 3. Positions — pick one to inspect

In [4]:
pos_array, xpos, ypos, npos, nshot = jiv.read_lp_positions(ifn)

pos_index = npos//2
print(f"\nInspecting position index {pos_index} of {npos} "
      f"(x={xpos[pos_index] if pos_index < len(xpos) else '?'} cm), {nshot} shots")

Using motion group: '<Hermes>    p29_LP'
Positions: 61 unique (x: 61, y: 1), 10 shots/position, 610 total shots.

Inspecting position index 30 of 61 (x=0.0 cm), 10 shots


## 4. Read raw I, V at this position

Shot-averaged V and per-shot I (scaled per the knobs).  Inspect the raw traces:
V should show the sweep ramps; I should be the probe current.

In [5]:
# pick which tip to look at in cells 4-7 (loop over `tips` for both probes)
tip = next(iter(tips))
I_chan, V_chan = tips[tip]["I"], tips[tip]["V"]
print(f"tip {tip}: I={I_chan}, V={V_chan}")

tarr, Vpos, Ipos = jiv.get_IV_at_position(
    run, scope_name, I_chan, V_chan, npos, nshot, pos_index)
print("Vpos:", Vpos.shape, " Ipos:", Ipos.shape, " tarr:", tarr.shape)

fig, ax = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
ax[0].plot(tarr * 1e3, Vpos)
ax[0].set_ylabel("V probe [V]")
ax[0].set_title(f"Raw traces — pos {pos_index}, tip {tip}")
for s in range(Ipos.shape[0]):
    ax[1].plot(tarr * 1e3, Ipos[s], lw=0.6, alpha=0.6)
ax[1].plot(tarr * 1e3, Ipos.mean(0), "k", lw=1.2, label="shot mean")
ax[1].set_ylabel("I [a.u.]"); ax[1].set_xlabel("t [ms]")
ax[1].legend(loc="upper right")
plt.tight_layout(); plt.show()

tip R: I=C3, V=C4
Vpos: (2500001,)  Ipos: (10, 2500001)  tarr: (2500001,)


## 5. Detect sweeps

`find_sweep_indices` locates each voltage ramp.  Tune `padding` if boundaries
look wrong; the overlay shows where each sweep starts/stops.

In [6]:
padding = 10   # <<< tune sweep boundary padding

start_ls, stop_ls = find_sweep_indices(Vpos, padding=padding)
print(f"Detected {len(start_ls)} sweeps")

plt.figure(figsize=(11, 3.5))
plt.plot(tarr * 1e3, Vpos, lw=0.8)
for a, b in zip(start_ls, stop_ls):
    plt.axvspan(tarr[a] * 1e3, tarr[b] * 1e3, color="orange", alpha=0.2)
plt.xlabel("t [ms]"); plt.ylabel("V probe [V]")
plt.title(f"Sweep windows ({len(start_ls)} found)")
plt.tight_layout(); plt.show()

Detected 11 sweeps


## 6. Reshape into per-sweep traces + smooth

`reshape_IV` standardizes sweep length and trims edges; current is smoothed.
Plot a few I–V traces to confirm shape and sign (electron branch should rise at
high V — flip `I_SIGN` in `Jun2026_IV.py` if inverted).

In [ ]:
trim_percent = 5    # <<< edge trim
smooth_sigma = 10   # <<< current smoothing

V_rs, I_rs = reshape_IV(Vpos[None, :], Ipos[None, :, :], start_ls, stop_ls, trim_percent)
I_rs = gaussian_filter1d(I_rs, smooth_sigma, axis=-1)

V_sweeps = V_rs[0]            # (nsweep, L)
I_sweeps = I_rs[0]           # (nshot, nsweep, L)
print("V_sweeps:", V_sweeps.shape, " I_sweeps:", I_sweeps.shape)

plt.figure(figsize=(8, 5))
for k in range(V_sweeps.shape[0]):
    plt.plot(V_sweeps[k], I_sweeps[:, k, :].mean(0), label=f"sweep {k}")
plt.xlabel("V probe [V]"); plt.ylabel("I [a.u.]")
plt.title(f"I-V traces (shot-averaged) - pos {pos_index}, tip {tip}")
plt.legend(); plt.tight_layout(); plt.show()

## 7. Analyze one I–V trace

`analyze_IV(V, I, plot=True)` extracts Vp / Te / ne and shows the fit.  Pick a
sweep index and shot to inspect the fit quality.

In [ ]:
sweep_idx = 1                  # <<< which sweep
I_trace = I_sweeps[:, sweep_idx, :].mean(0)   # shot-averaged current for this sweep
V_trace = V_sweeps[sweep_idx]

Vp, Te, ne = analyze_IV(V_trace, I_trace, plot=True)
print(f"Vp = {Vp:.2f} V,  Te = {Te:.2f} eV,  ne = {ne:.3e} cm^-3")

## 8. Both probes

Loop the read → detect → reshape → analyze steps over every complete-pair tip so
both probes are inspected at this position.  Missing probes stay blank.

In [ ]:
# analyze_tip_at_position bundles the read->detect->reshape->smooth->analyze
# steps walked manually in cells 4-7, so both probes use the same pipeline.
for t, ch in tips.items():
    Vp, Te, ne = jiv.analyze_tip_at_position(
        run, scope_name, ch["I"], ch["V"], npos, nshot, pos_index, sweep_idx,
        padding=padding, trim_percent=trim_percent, smooth_sigma=smooth_sigma)
    print(f"tip {t} (I={ch['I']}, V={ch['V']}): "
          f"Vp={Vp:.2f} V, Te={Te:.2f} eV, ne={ne:.3e} cm^-3")

for t in incomplete:
    print(f"tip {t}: BLANK (incomplete I+V pair)")

## 9. Interferometer line-averaged density (ne cross-check)

`interf_merge_lapd_daq.py` (bapsf_interferometer repo) merges two interferometer
traces per channel into the datarun file — the trace nearest the **first** shot
and the one nearest the **last** shot — under `diagnostics/interferometer/`.
Phase [rad] is converted to line-averaged density with each channel's
`calibration factor (m^-3/rad)` attribute (assumes 40 cm plasma length) and
plotted in cm⁻³ to compare with the IV-derived `ne` above.

Caveat for the calibration: the interferometer ne is a **chord average** along
the beam path, while the probe `ne` is **local** at the tip — compare the IV
value against the trace at the sweep's time in the discharge, and expect the
chord average to sit below the peak local density.

In [8]:
# Merged interferometer traces (nearest first + last shot of the run), read via
# the shared reader; phase (rad) -> line-averaged ne via each channel's
# calibration attribute.
from data_analysis.io import read_interferometer

interf = read_interferometer(ifn)

fig, axes = plt.subplots(len(interf), 1, sharex=True, squeeze=False,
                         figsize=(10, 2.8 * len(interf)))

for ax, (name, ch) in zip(axes[:, 0], interf.items()):
    for shot, reason in ch.skipped.items():
        print(f"  {name} shot {shot}: skipped ({reason})")
    for ne_line, shot, when in zip(ch.ne_line_cm3, ch.shots, ch.when):
        ax.plot(ch.t_ms, ne_line, label=f"shot {shot} ({when})")
    ax.set_ylabel(r"$\bar{n}_e$ [cm$^{-3}$]")
    ax.set_title(name)
    if ch.shots:
        ax.legend()

axes[-1, 0].set_xlabel("t [ms]")
plt.tight_layout()
plt.show()

  phase_p40 shot 1: skipped (scope offline)
  phase_p40 shot 610: skipped (scope offline)


## 10. Calibrate the ne profile against the interferometer

Wires the legacy `LP_analysis` recipe into the shared package: for each sweep,
the calibration factor is the interferometer line-averaged ne — averaged over
that sweep's **time window** and over the **two merged interferometer shots** —
divided by the chord average of the probe `ne(x)` profile
(`data_analysis.plasma.langmuir.interferometer_calibration`; generic — works
for any profile ∝ ne, e.g. Isat). Pick the chord with `INTERF_CHAN`.

The interferometer is on the true t=0 (plasma breakdown); the scope may trigger
at a different time. Set `T_OFFSET` (scope trigger time relative to breakdown)
to line the sweep windows up with the interferometer trace — it shifts only the
calibration windows and the comparison plot, never the scope time axes used
elsewhere in this notebook.

Needs the batch npz beside the raw HDF5 (run `jiv.process_run(ifn)` once), the
`interf` dict from section 9, and the section-5 sweep windows
(`start_ls` / `stop_ls`) for the timing.

In [ ]:
# Uses `interf` from section 9 and the sweep windows from section 5.
import os
import numpy as np

from data_analysis.plasma.langmuir import (
    interferometer_calibration, load_plasma_data, load_sweep_axes)
from data_analysis.utils import run_num_of

INTERF_CHAN = "phase_p29"   # <<< which interferometer chord to calibrate against
cal_tip = 'R'               # <<< probe tip whose batch npz to calibrate
T_OFFSET = 0.015              # <<< scope trigger time rel. to interferometer t=0
                            #     (breakdown) [s]; shifts only the calibration
                            #     windows, never the scope tarr

data_dir, run_num = os.path.dirname(ifn), run_num_of(ifn)
try:
    _, _, ne_arr, _, _, ne_err, t_ls = load_plasma_data(data_dir, run_num, tip=cal_tip)
    xpos_b, *_ = load_sweep_axes(data_dir, run_num, tip=cal_tip)
except FileNotFoundError as e:
    raise FileNotFoundError(
        f"Batch npz missing ({e}) -- run jiv.process_run(ifn) first.") from e

# per-sweep time windows from the section-5 sweep detection (scope time base)
assert len(start_ls) == ne_arr.shape[1], "sweep count mismatch vs section 5"
t_start, t_stop = tarr[start_ls], tarr[stop_ls]   # [s]

ch = interf[INTERF_CHAN]
ne_line_interf = ch.ne_line_avg_cm3()             # shot-averaged [cm^-3]
factor, ne_cal, chord = interferometer_calibration(
    ne_arr, xpos_b, t_start, t_stop, ch.t_ms * 1e-3, ne_line_interf,
    t_offset=T_OFFSET)

# probe sweep times on the interferometer base, for the comparison plot below
t_ls_i = t_ls + T_OFFSET

print(f"tip {cal_tip} vs {INTERF_CHAN} (interferometer shots {ch.shots}, averaged)")
for k, fac in enumerate(factor):
    print(f"  sweep {k:2d} @ t = {t_ls[k]*1e3:6.2f} ms (interf {t_ls_i[k]*1e3:6.2f} ms): "
          f"factor = {fac:.3f}")

fig, (ax_prof, ax_t) = plt.subplots(1, 2, figsize=(13, 5))

# left: raw (dashed) vs calibrated (solid) profiles at a few sweep times
show_sweeps = np.unique(np.linspace(0, ne_arr.shape[1] - 1, 4).astype(int))
for k in show_sweeps:
    c = plt.cm.viridis(k / max(ne_arr.shape[1] - 1, 1))
    ax_prof.plot(xpos_b, ne_arr[:, k], "--", color=c, lw=1)
    ax_prof.plot(xpos_b, ne_cal[:, k], "-", color=c, lw=1.6,
                 label=f"t = {t_ls[k]*1e3:.1f} ms (x{factor[k]:.2f})")
ax_prof.set_xlabel("x [cm]")
ax_prof.set_ylabel(r"$n_e$ [cm$^{-3}$]")
ax_prof.set_title(f"ne profiles, tip {cal_tip} -- dashed raw, solid calibrated")
ax_prof.legend()

# right: chord averages vs time + the per-sweep factor, on the interferometer base
ax_t.plot(ch.t_ms, ne_line_interf, "k", lw=1, label=f"{INTERF_CHAN} interferometer")
ax_t.plot(t_ls_i * 1e3, chord, "o-", label="probe chord avg (raw)")
ax_t.set_xlabel("t [ms] (interferometer base)")
ax_t.set_ylabel(r"$\bar{n}_e$ [cm$^{-3}$]")

ax_t.legend()

plt.tight_layout()
plt.show()

tip R vs phase_p29 (interferometer shots [1, 610], averaged)
  sweep  0 @ t =  -6.90 ms (interf   8.10 ms): factor = 3.986
  sweep  1 @ t =  -5.90 ms (interf   9.10 ms): factor = 3.612
  sweep  2 @ t =  -4.90 ms (interf  10.10 ms): factor = 3.601
  sweep  3 @ t =  -3.90 ms (interf  11.10 ms): factor = 3.559
  sweep  4 @ t =  -2.90 ms (interf  12.10 ms): factor = 3.548
  sweep  5 @ t =  -1.90 ms (interf  13.10 ms): factor = 3.459
  sweep  6 @ t =  -0.90 ms (interf  14.10 ms): factor = 3.467
  sweep  7 @ t =   0.10 ms (interf  15.10 ms): factor = 3.407
  sweep  8 @ t =   1.10 ms (interf  16.10 ms): factor = 3.374
  sweep  9 @ t =   2.10 ms (interf  17.10 ms): factor = 3.384
  sweep 10 @ t =   3.10 ms (interf  18.10 ms): factor = 3.221


In [ ]:
import numpy as np

# Runs grouped by gas-puff setting, written by group_by_gas_puff.py.
# {label: {"puff_v": volts, "puff_t": ms, "files": [hdf5 names...]}}
groups = np.load(
    r"D:\data\LAPD\jun2026-jia\gas_puff_groups.npy", allow_pickle=True
).item()

print(f"{len(groups)} gas-puff setting(s):")
for label, g in groups.items():
    print(f"\n  [{label}]  puff_v={g['puff_v']} V, puff_t={g['puff_t']} ms"
          f"  ({len(g['files'])} files)")
    for n in g["files"]:
        print(f"      {n}")